In [22]:
import json

# =========================
# 1. 读取 JS 文件
# =========================
with open("places_full.js", "r", encoding="utf-8") as f:
    content = f.read()

# =========================
# 2. 去掉 export 头
# =========================
# 把 "export const places =" 去掉
json_str = content.replace("export const places =", "").strip()

# 去掉最后的分号（如果有）
if json_str.endswith(";"):
    json_str = json_str[:-1]

# 转成 Python list
places = json.loads(json_str)


# =========================
# 3. 找异常数据
# =========================
problem_places = [
    p for p in places
    if (
        p.get("card_name") is None or
        (isinstance(p.get("card_name"), str) and p["card_name"].startswith("osm"))
    )
]

print("总数据量:", len(places))
print("异常数量:", len(problem_places))
print("示例:")
for p in problem_places[:5]:
    print(p["place_id"], p.get("card_name"), "→", p.get("name"))


# =========================
# 4. 修复逻辑
# =========================
for p in places:
    card = p.get("card_name")

    if (
        card is None or
        (isinstance(card, str) and card.startswith("osm"))
    ):
        if p.get("name"):
            p["card_name"] = p["name"]
        else:
            p["card_name"] = p["place_id"]


# =========================
# 5. 再检查一次
# =========================
after_problem = [
    p for p in places
    if (
        p.get("card_name") is None or
        (isinstance(p.get("card_name"), str) and p["card_name"].startswith("osm"))
    )
]

print("\n清洗后剩余异常:", len(after_problem))


# =========================
# 6. 写回 JS 文件
# =========================
output_str = "export const places = " + json.dumps(places, ensure_ascii=False, indent=2)

with open("places_clean.js", "w", encoding="utf-8") as f:
    f.write(output_str)

print("\n✅ 已输出 places_clean.js")

总数据量: 2945
异常数量: 146
示例:
osm_node_415803936 osm_node_415803936 → None
osm_node_611967784 osm_node_611967784 → None
osm_node_1647906579 osm_node_1647906579 → None
osm_node_4225998645 osm_node_4225998645 → None
osm_node_7612835933 osm_node_7612835933 → None

清洗后剩余异常: 146


In [23]:
# 是否存在 None
has_none = any(p.get("card_name") is None for p in places)

print("是否存在 None:", has_none)

是否存在 None: False


In [24]:
import json

# =========================
# 1. 读取 JS 文件
# =========================
with open("places_full.js", "r", encoding="utf-8") as f:
    content = f.read().strip()

prefix = "export const places ="
json_str = content[len(prefix):].strip()

if json_str.endswith(";"):
    json_str = json_str[:-1]

places = json.loads(json_str)


# =========================
# 2. 替换 osm_xxx → None
# =========================
def is_osm(name):
    return isinstance(name, str) and name.startswith("osm")

count_fix = 0

for p in places:
    if is_osm(p.get("card_name")):
        p["card_name"] = None   # 👉 关键：Python None → JS 会变成 null
        count_fix += 1

print("替换完成:", count_fix)


# =========================
# 3. 检查是否还有 osm
# =========================
remaining = [
    p for p in places
    if is_osm(p.get("card_name"))
]

print("剩余 osm:", len(remaining))


# =========================
# 4. 写回 JS 文件
# =========================
output = "export const places = " + json.dumps(places, ensure_ascii=False, indent=2)

with open("places_clean.js", "w", encoding="utf-8") as f:
    f.write(output)

print("✅ 已输出 places_clean.js")

替换完成: 146
剩余 osm: 0
✅ 已输出 places_clean.js
